In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models, datasets, transforms
import timm

In [2]:
import sys
print(sys.executable)

/home/iztihad/venvs/ml/bin/python


In [3]:
model_config = {
    "batch_size": 16,
    "input_size": 224,
    "architecture": "tiny-vision-transformer",
    "learning_rate": 0.001,
    "epochs": 20,
    "pretrained":True
}

In [4]:
data_transforms = {
    'train': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.Grayscale(num_output_channels=3),
        transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],
                             [0.229,0.224,0.225])
    ]),

    'val': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.Grayscale(num_output_channels=3),
        transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],
                             [0.229,0.224,0.225])
    ]),

    'test': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.Grayscale(num_output_channels=3),
        transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],
                             [0.229,0.224,0.225])
    ])
}

train_dir = "../BanglaLekha_dataset_split/train"
val_dir = "../BanglaLekha_dataset_split/validation"
test_dir = "../BanglaLekha_dataset_split/test"


train_dataset = datasets.ImageFolder(root=train_dir, transform=data_transforms["train"])
val_dataset = datasets.ImageFolder(root=val_dir, transform=data_transforms["val"])
test_dataset = datasets.ImageFolder(root=test_dir, transform=data_transforms["test"])

train_dataloader = DataLoader(train_dataset, batch_size=model_config["batch_size"], shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=model_config["batch_size"], shuffle=False)
test_dataloader = DataLoader(test_dataset, batch_size=model_config["batch_size"], shuffle=False)

In [5]:

efficientvit_b1 = timm.create_model("efficientvit_b1", pretrained=True)

for params in efficientvit_b1.parameters():
    params.requires_grad = False

efficientvit_b1.reset_classifier(84)


total_params = sum(p.numel() for p in efficientvit_b1.parameters())

gpu = torch.device("cuda")
efficientvit_b1 = efficientvit_b1.to(gpu)


model.safetensors:   0%|          | 0.00/36.5M [00:00<?, ?B/s]

In [6]:
print(efficientvit_b1)

EfficientVit(
  (stem): Stem(
    (in_conv): ConvNormAct(
      (dropout): Dropout(p=0.0, inplace=False)
      (conv): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (norm): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (act): Hardswish()
    )
    (res0): ResidualBlock(
      (pre_norm): Identity()
      (main): DSConv(
        (depth_conv): ConvNormAct(
          (dropout): Dropout(p=0.0, inplace=False)
          (conv): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=16, bias=False)
          (norm): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (act): Hardswish()
        )
        (point_conv): ConvNormAct(
          (dropout): Dropout(p=0.0, inplace=False)
          (conv): Conv2d(16, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (norm): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
       

In [7]:
print(total_params)

7635508


In [ ]:
import fine_tuning as ft

efficientvit_b1 = ft.fine_tune(model=efficientvit_b1, model_name="efficientvit_b1", state="full") #Change the state for fine tuning 

In [9]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW([
    {"params": tiny_vit.head.parameters(), "lr": 1e-3, "weight_decay": 1e-4},
    {"params": tiny_vit.stages.parameters(), "lr": 1e-5, "weight_decay": 1e-4},
    
])
epochs = model_config["epochs"]

In [10]:
def validate_model(model, val_dataloader):
    with torch.no_grad():
        model.eval()
        total = 0
        total_correct = 0

        for images, labels in val_dataloader:
            images = images.to(gpu)
            labels = labels.to(gpu)

            output = model(images)
            _, predicted = torch.max(output, 1)

            total = total + len(labels)
            total_correct = total_correct + (predicted == labels).sum().item()

        return total_correct/total 



In [20]:
def train_model(model, train_dataloader, val_dataloader, optimizer, criterion, epochs):
    
    max_val_accuracy = 0
    patience = 5
    count = 0
    for epoch in range(epochs):
        model.train()
        total_loss = 0

        for images, label in train_dataloader:
            images = images.to(gpu)
            label = label.to(gpu)

            optimizer.zero_grad()
            output = model(images)
            
            loss = criterion(output, label)
            loss.backward()
            optimizer.step()

            total_loss = total_loss + loss.item()
        
        val_accuracy = validate_model(tiny_vit, val_dataloader)

        if(val_accuracy > max_val_accuracy):
            max_val_accuracy = val_accuracy
            count = 0

            torch.save(model.state_dict(), "saved_parameters/tiny_ViT.pth")
        else:
            count = count + 1

        if(count >= patience):
            break
        

        print(f"Epoch: {epoch + 1}, Training Loss: {total_loss/len(train_dataloader)}, Validation Accuracy: {val_accuracy}")

In [16]:
train_model(tiny_vit, train_dataloader, val_dataloader, optimizer, criterion, model_config["epochs"])

Epoch: 1, Training Loss: 0.48512768334069095, Validation Accuracy: 0.9226129440858918
Epoch: 2, Training Loss: 0.3851851152228837, Validation Accuracy: 0.9320827552928403
Epoch: 3, Training Loss: 0.33911679429544167, Validation Accuracy: 0.9357621086917184
Epoch: 4, Training Loss: 0.30943464786689234, Validation Accuracy: 0.938054164907413
Epoch: 5, Training Loss: 0.27994213793986666, Validation Accuracy: 0.9399843175101031
Epoch: 6, Training Loss: 0.2625768740026563, Validation Accuracy: 0.9440255745219857
Epoch: 7, Training Loss: 0.24823744515980226, Validation Accuracy: 0.9422763737257976
Epoch: 8, Training Loss: 0.235440673184876, Validation Accuracy: 0.9448096990168285
Epoch: 9, Training Loss: 0.2243586440098072, Validation Accuracy: 0.9425779600699681
Epoch: 10, Training Loss: 0.21234790782452279, Validation Accuracy: 0.9475842933831956
Epoch: 11, Training Loss: 0.20182286445696165, Validation Accuracy: 0.9447493817479945
Epoch: 12, Training Loss: 0.19315396184754086, Validation 

In [21]:
tiny_vit.load_state_dict(torch.load("saved_parameters/tiny_ViT.pth"))
tiny_vit.eval()

TinyVit(
  (patch_embed): PatchEmbed(
    (conv1): ConvNorm(
      (conv): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (bn): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (act): GELU(approximate='none')
    (conv2): ConvNorm(
      (conv): Conv2d(32, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (bn): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
  )
  (stages): Sequential(
    (0): ConvLayer(
      (blocks): Sequential(
        (0): MBConv(
          (conv1): ConvNorm(
            (conv): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
            (bn): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          )
          (act1): GELU(approximate='none')
          (conv2): ConvNorm(
            (conv): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=256, bias=Fals

In [22]:
accuracy = validate_model(tiny_vit, test_dataloader)
print(f"Accuracy: {100 * accuracy}")

Accuracy: 94.31944486166601
